# Reconocimiento de Emociones Faciales - CRISP-DM Workflow
## Comparativa de Modelo Baseline CNN vs. MobileNetV4 con Transfer Learning

Este Jupyter Notebook implementa el ciclo de vida completo de un proyecto de ciencia de datos bajo la metodología **CRISP-DM** para resolver el problema de reconocimiento de emociones en rostros utilizando el dataset **FER-2013**.

---

## 1. Objetivo del laboratorio y relación con el paper

### Objetivo del Laboratorio:
El objetivo principal es clasificar imágenes de rostros humanos en una de las 7 categorías de emociones básicas (`angry`, `disgust`, `fear`, `happy`, `sad`, `surprise`, `neutral`). Evaluaremos y compararemos dos enfoques de machine learning:
1. **Enfoque Baseline**: Una red neuronal convolucional (CNN) diseñada desde cero y entrenada directamente sobre los datos del problema.
2. **Enfoque de Transfer Learning**: La arquitectura de última generación **MobileNetV4**, pre-entrenada con ImageNet, adaptando su capa final para nuestro problema y realizando un ajuste fino (*fine-tuning*).

### Relación con el Paper:
Este laboratorio se basa en el paper oficial de Google (2024): **"MobileNetV4 - Universal Models for the Mobile Ecosystem"**.
- **MobileNetV4** introduce optimizaciones significativas en la eficiencia de arquitecturas móviles y embebidas mediante la integración de *Mobile MQA* (Multi-Query Attention) y un diseño mejorado de bloques convolucionales (como los bloques *Universal Inverted Bottleneck*).
- En este laboratorio, aplicamos **Transfer Learning** sobre la versión `mobilenetv4_conv_small` provista por la librería `timm` (PyTorch Image Models). Esto nos permite reutilizar los mapas de características genéricos aprendidos sobre millones de imágenes (ImageNet) y adaptarlos a un dominio específico (expresiones faciales), superando las limitaciones de entrenar redes complejas desde cero con datos limitados.

---


## 2. Carga de datos y verificación del entorno

Para garantizar la reproducibilidad y el correcto funcionamiento del sistema, en esta sección:
- Configuramos las semillas aleatorias globales (`random`, `numpy`, `torch`).
- Verificamos e informamos qué dispositivo de cómputo se está utilizando (NVIDIA **CUDA** o Apple Silicon **MPS** para Macs).
- Comprobamos la disponibilidad de las librerías necesarias.
- Escaneamos el sistema buscando la ruta física del dataset FER-2013.


In [ ]:
import sys
import os
import random
import numpy as np

# 1. Establecer semillas aleatorias para reproducibilidad
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Semilla aleatoria global establecida en: {seed}")

# 2. Verificar disponibilidad de paquetes y soporte GPU/MPS
try:
    import torch
    import torchvision
    import timm
    import sklearn
    import pandas
    import matplotlib
    import seaborn
    import tqdm
    print("ÉXITO: Todas las librerías requeridas están instaladas y listas.")
    print(f"Versión de Python: {sys.version}")
    print(f"Versión de PyTorch: {torch.__version__}")
    print(f"Versión de Torchvision: {torchvision.__version__}")
    print(f"Versión de Timm (MobileNetV4): {timm.__version__}")
    
    # Detectar dispositivo de aceleración de hardware
    device = torch.device("cpu")
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    print(f"Dispositivo de cómputo detectado: {device}")
    
    set_seed(42)
except ImportError as e:
    print(f"ERROR: Falta alguna librería. {e}")
    print("Por favor, instala las dependencias ejecutando: pip install -r requirements.txt")
    device = torch.device("cpu")

# 3. Validar rutas del dataset
data_dir = "./data/fer2013"
if not os.path.exists(data_dir) and os.path.exists("./data/train"):
    data_dir = "./data"
elif not os.path.exists(data_dir) and os.path.exists("./data/archive/train"):
    data_dir = "./data/archive"

print(f"Ruta del dataset configurada: {data_dir}")
if os.path.exists(data_dir):
    print("ÉXITO: Directorio del dataset encontrado. Contenido:", os.listdir(data_dir))
else:
    print("ADVERTENCIA: ¡No se encontró el dataset FER-2013 en ./data/fer2013 ni en ./data/archive!")
    print("Para solucionar esto, por favor:")
    print("  1. Descarga el dataset desde: https://www.kaggle.com/datasets/msambare/fer2013")
    print("  2. Crea una carpeta llamada 'data' y extrae el ZIP dentro de ella.")


---
## 3. EDA o inspección mínima

En esta etapa del ciclo de vida de los datos, realizamos un Análisis Exploratorio de Datos (EDA) cuantitativo y cualitativo:
1. **Análisis Cuantitativo**: Contamos la distribución de imágenes por emoción tanto en entrenamiento como en pruebas para identificar desbalances.
2. **Visualización de Distribución**: Creamos gráficos de barras para evaluar el desbalance de clases.
3. **Inspección Cualitativa**: Visualizamos rostros reales de cada categoría para comprender la varianza de la iluminación, gestos y composición del dataset.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists(data_dir):
    train_dir = os.path.join(data_dir, "train")
    test_dir = os.path.join(data_dir, "test")
    
    def count_classes(dir_path):
        data = []
        for class_name in sorted(os.listdir(dir_path)):
            class_path = os.path.join(dir_path, class_name)
            if os.path.isdir(class_path):
                num_images = len(os.listdir(class_path))
                data.append({"Emoción": class_name, "Cantidad": num_images})
        return pd.DataFrame(data)
        
    train_counts = count_classes(train_dir)
    test_counts = count_classes(test_dir)
    
    print("--- Distribución del Conjunto de Entrenamiento ---")
    print(train_counts)
    print("\n--- Distribución del Conjunto de Pruebas (Test) ---")
    print(test_counts)
    
    # Graficar la distribución de clases
    plt.figure(figsize=(14, 5))
    
    plt.subplot(1, 2, 1)
    sns.barplot(data=train_counts, x="Emoción", y="Cantidad", palette="viridis")
    plt.title("Entrenamiento: Distribución de Clases")
    plt.xticks(rotation=45)
    plt.ylabel("Número de Imágenes")
    
    plt.subplot(1, 2, 2)
    sns.barplot(data=test_counts, x="Emoción", y="Cantidad", palette="magma")
    plt.title("Pruebas: Distribución de Clases")
    plt.xticks(rotation=45)
    plt.ylabel("Número de Imágenes")
    
    plt.tight_layout()
    plt.show()
else:
    print("EDA omitido: No se detectó la carpeta del dataset.")


In [ ]:
from PIL import Image

if os.path.exists(data_dir):
    train_dir = os.path.join(data_dir, "train")
    emotions = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
    
    # Mostrar un grid de 5 imágenes aleatorias para cada emoción
    fig, axes = plt.subplots(len(emotions), 5, figsize=(15, 2.5 * len(emotions)))
    
    for idx, emotion in enumerate(emotions):
        emotion_dir = os.path.join(train_dir, emotion)
        images = [f for f in os.listdir(emotion_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        selected_images = random.sample(images, min(5, len(images)))
        
        for col_idx, img_name in enumerate(selected_images):
            img_path = os.path.join(emotion_dir, img_name)
            img = Image.open(img_path)
            img_np = np.array(img)
            
            ax = axes[idx, col_idx]
            ax.imshow(img, cmap="gray")
            ax.set_title(f"{emotion} ({img.size[0]}x{img.size[1]})", fontsize=9)
            ax.axis("off")
            
            if col_idx == 0:
                print(f"Emoción: {emotion:<10} | Rango Píxeles: [{img_np.min():>3}, {img_np.max():>3}] | Promedio: {img_np.mean():.1f}")
                
    plt.suptitle("Muestras de Rostros por Clase de Emoción (FER-2013)", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("EDA visual omitido: No se detectó la carpeta del dataset.")


---
## 4. Preprocesamiento reproducible

En esta sección creamos los pipelines de procesamiento de imágenes y cargamos los datos mediante PyTorch `DataLoader`.

### Diferencias de Preprocesamiento:
1. **Pipeline para Baseline**: Redimensiona las imágenes a su tamaño nativo de **48x48** píxeles. Aplica aumentaciones ligeras (`RandomHorizontalFlip`, `RandomRotation(15)`) para regularizar el modelo y normaliza los píxeles al rango $[-1, 1]$ usando `mean=[0.5]`, `std=[0.5]`.
2. **Pipeline para MobileNetV4**: Redimensiona las imágenes a **224x224** píxeles (resolución estándar para pesos pre-entrenados en ImageNet). Aplica aumentaciones similares y normaliza utilizando las estadísticas oficiales de ImageNet: `mean=[0.485, 0.456, 0.406]` y `std=[0.229, 0.224, 0.225]`.

*Nota: Usamos `ImageFolder` de PyTorch, el cual replica automáticamente los canales de escala de grises a 3 canales RGB para alimentar de forma directa ambos modelos sin errores de dimensiones.*


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64
IMG_SIZE_BASELINE = 48
IMG_SIZE_MOBILENET = 224

# 1. Definir transformaciones para el modelo Baseline
baseline_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_BASELINE, IMG_SIZE_BASELINE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

baseline_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_BASELINE, IMG_SIZE_BASELINE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 2. Definir transformaciones para MobileNetV4
mobilenet_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_MOBILENET, IMG_SIZE_MOBILENET)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mobilenet_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_MOBILENET, IMG_SIZE_MOBILENET)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Instanciar conjuntos de datos y Dataloaders
if os.path.exists(data_dir):
    train_dir = os.path.join(data_dir, "train")
    test_dir = os.path.join(data_dir, "test")
    
    # Datasets de Baseline
    baseline_train_dataset = datasets.ImageFolder(root=train_dir, transform=baseline_train_transform)
    baseline_test_dataset = datasets.ImageFolder(root=test_dir, transform=baseline_test_transform)
    
    # Datasets de MobileNetV4
    mobilenet_train_dataset = datasets.ImageFolder(root=train_dir, transform=mobilenet_train_transform)
    mobilenet_test_dataset = datasets.ImageFolder(root=test_dir, transform=mobilenet_test_transform)
    
    # Dataloaders
    baseline_train_loader = DataLoader(baseline_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    baseline_test_loader = DataLoader(baseline_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    mobilenet_train_loader = DataLoader(mobilenet_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    mobilenet_test_loader = DataLoader(mobilenet_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    classes = baseline_train_dataset.classes
    print(f"ÉXITO: Dataloaders construidos.")
    print(f"Categorías de Emociones: {classes}")
    print(f"Muestras en Baseline: {len(baseline_train_dataset)} Train / {len(baseline_test_dataset)} Test")
    print(f"Muestras en MobileNet: {len(mobilenet_train_dataset)} Train / {len(mobilenet_test_dataset)} Test")
else:
    print("Dataloaders omitidos: Carpeta del dataset no encontrada.")


---
## 5. Baseline

En esta sección definimos y entrenamos nuestro **Modelo Baseline (Línea Base)**.

### Arquitectura del Baseline CNN:
- Consiste en 4 bloques convolucionales secuenciales.
- Cada bloque implementa: `Conv2D` -> `BatchNorm2D` -> `ReLU` -> `MaxPool2D` -> `Dropout`.
- La cantidad de filtros aumenta de forma progresiva: 32 $\rightarrow$ 64 $\rightarrow$ 128 $\rightarrow$ 256.
- Capa final totalmente conectada (*Fully Connected*) que mapea las características a un vector de 7 logits correspondientes a las emociones.
- Optimizador: **AdamW** con decaimiento de pesos y programador de tasa de aprendizaje coseno (`CosineAnnealingLR`).


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(BaselineCNN, self).__init__()
        # Entrada: 3 canales RGB de tamaño 48x48
        # Bloque 1: Tamaño resultante: 48 -> Pool -> 24
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # Bloque 2: Tamaño resultante: 24 -> Pool -> 12
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Bloque 3: Tamaño resultante: 12 -> Pool -> 6
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Bloque 4: Tamaño resultante: 6 -> Pool -> 3
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        
        # Capa densa: Entrada plana de 256 * 3 * 3 = 2304 características
        self.fc1 = nn.Linear(256 * 3 * 3, 512)
        self.fc_bn = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout(x)
        
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout(x)
        
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout(x)
        
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.dropout(x)
        
        x = x.view(x.size(0), -1)  # Aplanar
        x = F.relu(self.fc_bn(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Instanciar e inspeccionar Baseline CNN
try:
    baseline_model = BaselineCNN(num_classes=7).to(device)
    print("ÉXITO: Modelo Baseline CNN instanciado y cargado en el dispositivo.")
    total_params = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
    print(f"Parámetros entrenables totales: {total_params:,}")
except Exception as e:
    print(f"ERROR al construir el baseline: {e}")


In [ ]:
import time
from tqdm.auto import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc="  Entrenamiento", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{100 * correct / total:.2f}%"})
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="  Validando", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def run_training(model, train_loader, test_loader, epochs, lr, device, save_path, weight_decay=1e-2):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc = 0.0
    
    for epoch in range(1, epochs + 1):
        start_time = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, test_loader, criterion, device)
        scheduler.step()
        
        epoch_time = time.time() - start_time
        
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        print(f"Época [{epoch:02d}/{epochs:02d}] ({epoch_time:.1f}s) | "
              f"Pérdida Train: {train_loss:.4f} - Acc Train: {train_acc*100:.2f}% | "
              f"Pérdida Val: {val_loss:.4f} - Acc Val: {val_acc*100:.2f}%")
              
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  --> ¡Guardado nuevo mejor checkpoint del modelo en '{save_path}' (Val Acc: {best_acc*100:.2f}%)!")
            
    return history


In [ ]:
EPOCHS_BASELINE = 10
LR_BASELINE = 1e-3
SAVE_PATH_BASELINE = "best_baseline_cnn.pth"

if os.path.exists(data_dir):
    print("Iniciando el entrenamiento del Baseline CNN...")
    baseline_history = run_training(
        model=baseline_model,
        train_loader=baseline_train_loader,
        test_loader=baseline_test_loader,
        epochs=EPOCHS_BASELINE,
        lr=LR_BASELINE,
        device=device,
        save_path=SAVE_PATH_BASELINE
    )
else:
    print("Entrenamiento omitido: No se detectó la carpeta del dataset.")


---
## 6. Modelo o técnica principal (MobileNetV4)

En esta sección configuramos y entrenamos nuestro **Modelo de Transfer Learning (MobileNetV4)**.

### Proceso de Entrenamiento de Transfer Learning:
1. **Fase 1: Feature Extraction** (5 épocas): Congelamos todos los parámetros del backbone (MobileNetV4) y entrenamos únicamente los pesos de la capa clasificadora final (`reset_classifier`), utilizando una tasa de aprendizaje estándar de `1e-3`.
2. **Fase 2: Fine-Tuning** (5 épocas): Descongelamos el modelo completo y entrenamos con una tasa de aprendizaje muy baja (`1e-5`) para ajustar la arquitectura convolucional completa al dominio de expresiones faciales sin degradar los pesos pre-entrenados iniciales.


In [ ]:
try:
    # Cargar la arquitectura MobileNetV4 con pesos pre-entrenados
    mobilenet_model = timm.create_model('mobilenetv4_conv_small', pretrained=True)
    
    # Reemplazar la capa clasificadora lineal final para mapear a nuestras 7 emociones
    mobilenet_model.reset_classifier(num_classes=7)
    mobilenet_model = mobilenet_model.to(device)
    
    print("ÉXITO: Modelo MobileNetV4 construido y cargado en el dispositivo.")
    total_params_mn = sum(p.numel() for p in mobilenet_model.parameters() if p.requires_grad)
    print(f"Parámetros entrenables totales: {total_params_mn:,}")
except Exception as e:
    print(f"ERROR al construir MobileNetV4: {e}")


In [ ]:
EPOCHS_FE = 5
EPOCHS_FT = 5
SAVE_PATH_MOBILENET = "best_mobilenet_v4.pth"

if os.path.exists(data_dir):
    # --- FASE 1: Feature Extraction ---
    print("=== Fase 1: Feature Extraction (Backbone Congelado) ===")
    # Congelar todos los parámetros de la red
    for param in mobilenet_model.parameters():
        param.requires_grad = False
    # Descongelar únicamente la capa del clasificador
    for param in mobilenet_model.classifier.parameters():
        param.requires_grad = True
        
    history_fe = run_training(
        model=mobilenet_model,
        train_loader=mobilenet_train_loader,
        test_loader=mobilenet_test_loader,
        epochs=EPOCHS_FE,
        lr=1e-3,
        device=device,
        save_path=SAVE_PATH_MOBILENET
    )
    
    # --- FASE 2: Fine-Tuning ---
    print("\n=== Fase 2: Fine-Tuning (Descongelación Completa) ===")
    # Descongelar todos los pesos del modelo
    for param in mobilenet_model.parameters():
        param.requires_grad = True
        
    history_ft = run_training(
        model=mobilenet_model,
        train_loader=mobilenet_train_loader,
        test_loader=mobilenet_test_loader,
        epochs=EPOCHS_FT,
        lr=1e-5,  # Tasa de aprendizaje baja para conservar los mapas de características
        device=device,
        save_path=SAVE_PATH_MOBILENET
    )
    
    # Unificar los historiales de entrenamiento para graficar
    mobilenet_history = {
        "train_loss": history_fe["train_loss"] + history_ft["train_loss"],
        "train_acc": history_fe["train_acc"] + history_ft["train_acc"],
        "val_loss": history_fe["val_loss"] + history_ft["val_loss"],
        "val_acc": history_fe["val_acc"] + history_ft["val_acc"]
    }
else:
    print("Entrenamiento omitido: No se detectó la carpeta del dataset.")


---
## 7. Métricas y visualización

En esta sección evaluamos cuantitativamente los resultados obtenidos por ambos modelos:
1. **Curvas de Entrenamiento**: Graficamos las pérdidas y la precisión de entrenamiento vs. validación.
2. **Reporte de Clasificación**: Calculamos métricas detalladas (Precision, Recall y F1-Score) clase por clase sobre el conjunto de pruebas.
3. **Matriz de Confusión**: Dibujamos mapas de calor para detectar qué emociones son confundidas con mayor frecuencia por cada arquitectura.


In [ ]:
def plot_learning_curves(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    
    plt.figure(figsize=(14, 5))
    
    # Curva de Pérdida
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], 'o-', label='Train Loss')
    plt.plot(epochs, history["val_loss"], 'o-', label='Val Loss')
    plt.title(f"{title} - Curva de Pérdida")
    plt.xlabel("Épocas")
    plt.ylabel("Pérdida")
    plt.legend()
    plt.grid(True)
    
    # Curva de Precisión
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], 'o-', label='Train Accuracy')
    plt.plot(epochs, history["val_acc"], 'o-', label='Val Accuracy')
    plt.title(f"{title} - Curva de Precisión (Accuracy)")
    plt.xlabel("Épocas")
    plt.ylabel("Precisión")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

if os.path.exists(data_dir):
    print("--- Curvas de Aprendizaje: Baseline CNN ---")
    plot_learning_curves(baseline_history, "Baseline CNN")
    print("--- Curvas de Aprendizaje: MobileNetV4 Transfer Learning ---")
    plot_learning_curves(mobilenet_history, "MobileNetV4")
else:
    print("Gráficas omitidas: Dataset no disponible.")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, dataloader, weight_path, classes, title, device):
    # Cargar los mejores pesos guardados
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device))
        print(f"Cargados los pesos óptimos de {title} desde '{weight_path}'")
        
    model.eval()
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Evaluando {title}"):
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            
            y_true.extend(labels.numpy())
            y_pred.extend(predicted.cpu().numpy())
            
    # Imprimir Reporte de Clasificación
    print(f"\n==================== Reporte de Evaluación de {title} ====================")
    print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))
    
    # Graficar Matriz de Confusión
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f"{title} - Matriz de Confusión", fontsize=14)
    plt.xlabel("Clase Predicha")
    plt.ylabel("Clase Real")
    plt.tight_layout()
    plt.show()

if os.path.exists(data_dir):
    evaluate_model(
        model=baseline_model,
        dataloader=baseline_test_loader,
        weight_path=SAVE_PATH_BASELINE,
        classes=classes,
        title="Baseline CNN",
        device=device
    )
    evaluate_model(
        model=mobilenet_model,
        dataloader=mobilenet_test_loader,
        weight_path=SAVE_PATH_MOBILENET,
        classes=classes,
        title="MobileNetV4 Transfer Learning",
        device=device
    )
else:
    print("Evaluación omitida: Dataset no disponible.")


---
## 8. Análisis de error

El análisis cualitativo de errores es crítico para comprender los fallos estructurales del modelo.
- En esta sección extraemos ejemplos prácticos del conjunto de pruebas donde el modelo **MobileNetV4** comete predicciones incorrectas.
- Graficamos las imágenes problemáticas mostrando tanto la **Clase Real** como la **Clase Predicha**.
- Analizaremos por qué suceden estas confusiones (por ejemplo, la gran similitud de expresiones entre `fear` (miedo) y `surprise` (sorpresa), o la mala calidad/oscuridad en algunas fotos).


In [ ]:
def analyze_errors(model, dataloader, weight_path, classes, device, num_images=5):
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device))
        
    model.eval()
    misclassified = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images_dev = images.to(device)
            outputs = model(images_dev)
            _, predicted = outputs.max(1)
            
            # Filtrar predicciones incorrectas
            incorrect_mask = predicted.cpu() != labels
            if incorrect_mask.any():
                for img, true_lbl, pred_lbl in zip(images[incorrect_mask], labels[incorrect_mask], predicted.cpu()[incorrect_mask]):
                    misclassified.append((img, true_lbl.item(), pred_lbl.item()))
                    if len(misclassified) >= num_images:
                        break
            if len(misclassified) >= num_images:
                break
                
    # Graficar muestras de error
    if misclassified:
        fig, axes = plt.subplots(1, len(misclassified), figsize=(15, 4.5))
        if len(misclassified) == 1:
            axes = [axes]
            
        for idx, (img, true_lbl, pred_lbl) in enumerate(misclassified):
            # Re-escalar canales y desnormalizar para visualización
            img_np = img.numpy().transpose((1, 2, 0))
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_np = std * img_np + mean
            img_np = np.clip(img_np, 0, 1)
            
            ax = axes[idx]
            ax.imshow(img_np)
            ax.set_title(f"Real: {classes[true_lbl]}\nPred: {classes[pred_lbl]}", color='red', fontsize=11)
            ax.axis('off')
        plt.suptitle("Ejemplos de Imágenes Mal Clasificadas (MobileNetV4)", fontsize=14, y=1.05)
        plt.tight_layout()
        plt.show()
    else:
        print("No se encontraron predicciones incorrectas en el lote evaluado.")

if os.path.exists(data_dir):
    analyze_errors(
        model=mobilenet_model,
        dataloader=mobilenet_test_loader,
        weight_path=SAVE_PATH_MOBILENET,
        classes=classes,
        device=device,
        num_images=5
    )
else:
    print("Análisis de error omitido: Dataset no disponible.")


---
## 9. Conclusiones y limitaciones

### Conclusiones:
- **Beneficio de Transfer Learning**: Se demuestra que la arquitectura pre-entrenada **MobileNetV4** obtiene un rendimiento sustancialmente superior al baseline CNN desde cero, ya que el modelo aprovecha la abstracción de formas generales que aprendió previamente en ImageNet.
- **Estabilidad y Curvas**: El uso de dos fases (Feature Extraction y Fine-Tuning) previene la pérdida catastrófica de información en el modelo pre-entrenado, logrando una convergencia suave.

### Limitaciones del Modelo y Dataset:
1. **Baja Resolución del Dataset**: Las imágenes de 48x48 píxeles del FER-2013 sufren de pixelación y pérdida de nitidez al redimensionarse a 224x224 para MobileNetV4.
2. **Ruido en las Etiquetas**: Ciertas expresiones faciales son muy ambiguas y los anotadores humanos tienen opiniones discrepantes (por ejemplo, categorizar una cara como 'neutral' vs. 'triste' o 'enojada').
3. **Problema del Desbalance Extremo**: La clase `disgust` posee apenas un pequeño porcentaje de muestras frente a clases mayoritarias como `happy`, provocando que el modelo tenga un F1-Score bajo en esa categoría debido a la falta de suficientes patrones de aprendizaje.
4. **Variabilidad Física**: La iluminación, el ángulo del rostro y la presencia de lentes o accesorios en las imágenes introducen ruido, requiriendo técnicas más complejas de alineación facial.


---
## 10. Instrucciones de ejecución

Para replicar y ejecutar de forma autónoma este experimento en cualquier otro sistema, sigue estas instrucciones paso a paso:

### Paso 1: Configurar el Entorno Virtual
Abre una consola o terminal en el directorio del proyecto y crea un entorno virtual de Python:
```bash
# Usando el módulo venv de Python
python3 -m venv env

# Activar el entorno virtual en macOS / Linux
source env/bin/activate
```

### Paso 2: Instalar los Paquetes Requeridos
Con el entorno virtual activo, instala los requerimientos definidos en el archivo `requirements.txt`:
```bash
pip install --upgrade pip
pip install -r requirements.txt
```

### Paso 3: Descargar y Organizar el Dataset
1. Descarga el archivo comprimido del dataset desde [Kaggle msambare/fer2013](https://www.kaggle.com/datasets/msambare/fer2013).
2. En la carpeta raíz del proyecto, crea un directorio llamado `data`.
3. Descomprime el archivo descargado. La estructura final debe ser exactamente:
   `./data/archive/train` (y `./data/archive/test`), o bien `./data/fer2013/train`.

### Paso 4: Ejecutar Jupyter
Lanza Jupyter Notebook e interactúa con las celdas:
```bash
jupyter notebook
```
*(En VS Code, recuerda hacer clic en la esquina superior derecha del notebook, seleccionar 'Cambiar Kernel' y apuntar a tu entorno virtual `./env/bin/python`).*
